# Figure 1 — The replicate-aware evaluation protocol
**Panel A** protocol schematic (drawn). **Panel B** same-half vs
independent-half scoring across models in K562 essential — the gap is the
leakage component the protocol removes. Data: `j22_k562_foldfrozen_centroid.csv`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RES = os.path.join("..", "..", "results")
FIG = os.path.join("..", "..", "figures")
os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 300,
                     "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False})
MODELS = ["lw", "svd20", "moppca5", "nn10", "nn50"]
MODEL_LABELS = {"lw": "LW shrinkage", "svd20": "SVD-20",
                "moppca5": "moPPCA-5", "nn10": "centroid-10",
                "nn50": "centroid-50"}
CONTEXTS = [
    ("K562 essential", "j22_k562_foldfrozen_centroid.csv"),
    ("K562 GWPS", "j22_k562_gwps_foldfrozen_centroid.csv"),
    ("Norman", "j49_norman_foldfrozen.csv"),
    ("RPE1", "j49_rpe1_foldfrozen.csv"),
    ("sci-Plex3", "j49_sciplex3.csv"),
]
def load_ctx(name, fname):
    df = pd.read_csv(os.path.join(RES, fname))
    df["context"] = name
    return df


# ---- Panel A: schematic ----
fig, ax = plt.subplots(figsize=(7.2, 2.6))
ax.axis("off")
boxes = [
    (0.02, "Perturbation\nhalves A | B", "#dbeafe"),
    (0.26, "K-fold split\n(genes + models\nfit in-train)", "#dcfce7"),
    (0.50, "Predict masked 10%\nfrom observed 90%", "#fef9c3"),
    (0.74, "Score vs A (same half)\nScore vs B (independent)", "#fee2e2"),
]
for x, txt, c in boxes:
    ax.add_patch(plt.Rectangle((x, 0.25), 0.20, 0.5, fc=c, ec="0.4"))
    ax.text(x + 0.10, 0.5, txt, ha="center", va="center", fontsize=8)
for x in [0.22, 0.46, 0.70]:
    ax.annotate("", xy=(x + 0.04, 0.5), xytext=(x, 0.5),
                arrowprops=dict(arrowstyle="->", color="0.3"))
ax.set_title("Replicate-resolved masked completion", fontsize=10)
fig.savefig(os.path.join(FIG, "fig1a_schematic.png"), bbox_inches="tight")
plt.show()


In [ ]:

# ---- Panel B: same-half vs independent-half (K562 essential) ----
df = pd.read_csv(os.path.join(RES, "j22_k562_foldfrozen_centroid.csv"))
rows = []
for m in MODELS:
    rows.append({"model": MODEL_LABELS[m],
                 "same half": df[f"{m}_A_cen"].mean(),
                 "independent half": df[f"{m}_B_cen"].mean()})
t = pd.DataFrame(rows).set_index("model")
ax = t.plot.bar(figsize=(7.2, 3.2), color=["#93c5fd", "#1d4ed8"])
ax.set_ylabel("centered correlation")
ax.set_title("Same-half scores exceed independent-half scores for every model\n"
             "(K562 essential, n=%d perturbations)" % len(df))
ax.legend(frameon=False)
plt.xticks(rotation=0)
fig2 = ax.get_figure()
fig2.savefig(os.path.join(FIG, "fig1b_same_vs_indep.png"), bbox_inches="tight")
plt.show()
infl = (t["same half"] - t["independent half"]).round(3)
print("inflation per model:", dict(infl))
